## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Plot Styling ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'axes.titleweight': 'bold',
})
PALETTE = sns.color_palette('Set2')
RED     = '#E74C3C'
BLUE    = '#2E86AB'
GRAY    = '#555555'

print('Libraries loaded ✅')

In [ ]:
df = pd.read_csv('/kaggle/input/airline-delay/Airline_Delay_Cause.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 2. Data Cleaning & Preprocessing <a id='2'></a>

In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# ── Fill numeric nulls with column mean (< 0.3% missing — safe to impute) ────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# ── Convert month integer → month name ───────────────────────────────────────
df['month_name'] = pd.to_datetime(df['month'], format='%m').dt.month_name()

# ── Keep numeric month for time-based sorting ─────────────────────────────────
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
df['month_name'] = pd.Categorical(df['month_name'], categories=MONTH_ORDER, ordered=True)

print('Cleaning complete ✅')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

In [ ]:
# ── Basic Statistics ──────────────────────────────────────────────────────────
df.describe().T.style.background_gradient(cmap='Blues', axis=1)

## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

In [ ]:
# ── 3.1 Delay Distribution ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['arr_del15'].dropna(), bins=80, color=BLUE, edgecolor='white', alpha=0.9)
axes[0].set_title('Distribution of Delayed Flights (arr_del15)')
axes[0].set_xlabel('Number of Delayed Flights')
axes[0].set_ylabel('Frequency')

df['delay_ratio'] = df['arr_del15'] / df['arr_flights']
axes[1].hist(df['delay_ratio'].dropna(), bins=80, color=RED, edgecolor='white', alpha=0.9)
axes[1].set_title('Distribution of Delay Ratio (arr_del15 / arr_flights)')
axes[1].set_xlabel('Delay Ratio')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Correlation Heatmap ───────────────────────────────────────────────────
delay_vars = ['arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
              'security_ct', 'late_aircraft_ct', 'arr_flights',
              'arr_cancelled', 'arr_diverted']

corr = df[delay_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, center=0, square=True)
plt.title('Correlation Matrix — Delay Components')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Yearly Trend ──────────────────────────────────────────────────────────
yearly = df.groupby('year')['delay_ratio'].mean()

plt.figure(figsize=(12, 5))
ax = plt.gca()
ax.fill_between(yearly.index, yearly.values, alpha=0.15, color=BLUE)
ax.plot(yearly.index, yearly.values, marker='o', color=BLUE, linewidth=2.5)

for x, y in zip(yearly.index, yearly.values):
    ax.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=9, color=GRAY)

ax.set_title('Average Delay Ratio by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Delay Ratio')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Monthly Seasonality ───────────────────────────────────────────────────
monthly = df.groupby('month_name', observed=True)['delay_ratio'].mean()

plt.figure(figsize=(12, 5))
colors = [RED if v == monthly.max() else BLUE for v in monthly.values]
bars = plt.bar(monthly.index, monthly.values, color=colors, edgecolor='white', linewidth=0.8)
plt.title('Average Delay Ratio by Month (All Years)')
plt.xlabel('Month')
plt.ylabel('Avg Delay Ratio')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Delay Component Breakdown (Overall) ───────────────────────────────────
delay_components = ['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct']
comp_means = df[delay_components].mean()
comp_pct   = comp_means / comp_means.sum() * 100

LABEL_MAP = {
    'carrier_ct': 'Carrier',
    'weather_ct': 'Weather',
    'nas_ct': 'NAS',
    'security_ct': 'Security',
    'late_aircraft_ct': 'Late Aircraft'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
axes[0].barh([LABEL_MAP[c] for c in delay_components], comp_pct.values,
             color=sns.color_palette('Set2', 5))
for i, v in enumerate(comp_pct.values):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=10)
axes[0].set_title('Delay Cause Breakdown (%)')
axes[0].set_xlabel('Percentage')

# Pie chart
axes[1].pie(comp_pct.values,
            labels=[LABEL_MAP[c] for c in delay_components],
            autopct='%1.1f%%',
            colors=sns.color_palette('Set2', 5),
            startangle=140)
axes[1].set_title('Delay Cause — Pie Chart')

plt.suptitle('Overall Delay Cause Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6 Heatmap: Delay Ratio by Year × Month ─────────────────────────────────
pivot_ym = df.groupby(['year', 'month_name'], observed=True)['delay_ratio'].mean().unstack()

plt.figure(figsize=(14, 7))
sns.heatmap(pivot_ym, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.4, linecolor='white', cbar_kws={'label': 'Delay Ratio'})
plt.title('Average Delay Ratio — Year × Month')
plt.xlabel('Month')
plt.ylabel('Year')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.7 Cancelled & Diverted Flights Over Time ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, color, label in zip(
    axes,
    ['arr_cancelled', 'arr_diverted'],
    [RED, BLUE],
    ['Cancellations', 'Diversions']
):
    yearly_col = df.groupby('year')[col].mean()
    ax.fill_between(yearly_col.index, yearly_col.values, alpha=0.15, color=color)
    ax.plot(yearly_col.index, yearly_col.values, marker='o', color=color, linewidth=2.5)
    ax.set_title(f'Avg {label} by Year')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'Avg {label}')

plt.tight_layout()
plt.show()

## 4. Deep-Dive: Airport & Carrier Analysis <a id='4'></a>

In [ ]:
# ── 4.1 Top 10 Airports by Delay Ratio ───────────────────────────────────────
airport_delay = (df.groupby('airport_name')['delay_ratio']
                   .mean()
                   .sort_values(ascending=False)
                   .head(10))

plt.figure(figsize=(13, 6))
bars = plt.barh(airport_delay.index[::-1], airport_delay.values[::-1],
                color=sns.color_palette('Reds_r', 10))
for bar, val in zip(bars, airport_delay.values[::-1]):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.title('Top 10 Airports by Average Delay Ratio')
plt.xlabel('Average Delay Ratio')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2 Airport Delay Heatmap (lowest avg delay — smallest airports) ──────────
top15_airports = (df.groupby('airport_name')['delay_ratio']
                    .mean()
                    .sort_values(ascending=False)
                    .head(15)
                    .index)

pivot_airport = (df[df['airport_name'].isin(top15_airports)]
                 .groupby(['airport_name', 'year'])['delay_ratio']
                 .mean()
                 .unstack())

# Shorten labels for readability
pivot_airport.index = pivot_airport.index.str.split(':').str[0].str.strip()

plt.figure(figsize=(14, 7))
sns.heatmap(pivot_airport, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.3, linecolor='white')
plt.title('Delay Ratio Heatmap — Top 15 Airports × Year')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3 Top 10 Carriers by Delay Ratio ───────────────────────────────────────
carrier_delay = (df.groupby('carrier_name')['delay_ratio']
                   .mean()
                   .sort_values(ascending=False)
                   .head(10))

plt.figure(figsize=(12, 6))
sns.barplot(x=carrier_delay.values, y=carrier_delay.index,
            palette='Blues_r')
plt.title('Top 10 Carriers by Average Delay Ratio')
plt.xlabel('Average Delay Ratio')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.4 Delay Cause Breakdown per Carrier (Stacked) ──────────────────────────
carrier_comp = (df.groupby('carrier_name')[delay_components]
                  .sum()
                  .sort_values('late_aircraft_ct', ascending=False)
                  .head(12))

carrier_comp.rename(columns=LABEL_MAP).plot(
    kind='barh', stacked=True, figsize=(13, 8),
    color=sns.color_palette('Set2', 5))
plt.title('Delay Component Breakdown by Carrier (Top 12)')
plt.xlabel('Total Delay Count')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5 American Airlines (AA) Deep Dive ─────────────────────────────────────
aa = df[df['carrier'] == 'AA'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Delay cause pie
aa_comp = aa[delay_components].mean()
axes[0].pie(aa_comp.values,
            labels=[LABEL_MAP[c] for c in delay_components],
            autopct='%1.1f%%',
            colors=sns.color_palette('Set2', 5), startangle=140)
axes[0].set_title('American Airlines — Delay Causes')

# Yearly delay
aa_yr = aa.groupby('year')['delay_ratio'].mean()
axes[1].fill_between(aa_yr.index, aa_yr.values, alpha=0.15, color=RED)
axes[1].plot(aa_yr.index, aa_yr.values, marker='o', color=RED, linewidth=2.5)
axes[1].set_title('AA — Yearly Avg Delay Ratio')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Avg Delay Ratio')

# Monthly delay
aa_mo = aa.groupby('month_name', observed=True)['delay_ratio'].mean()
axes[2].fill_between(range(len(aa_mo)), aa_mo.values, alpha=0.15, color=BLUE)
axes[2].plot(range(len(aa_mo)), aa_mo.values, marker='o', color=BLUE, linewidth=2.5)
axes[2].set_xticks(range(len(aa_mo)))
axes[2].set_xticklabels(aa_mo.index, rotation=30, ha='right', fontsize=8)
axes[2].set_title('AA — Monthly Avg Delay Ratio')

plt.suptitle('American Airlines (AA) Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. COVID-19 Period Analysis (2019–2021) <a id='5'></a>

In [ ]:
# ── 5.1 Delay/Cancelled/Diverted — 2019 vs 2020 vs 2021 ──────────────────────
covid_years = [2019, 2020, 2021]
metrics = ['arr_del15', 'arr_cancelled', 'arr_diverted', 'arr_flights']

covid_summary = []
for yr in covid_years:
    row = df[df['year'] == yr][metrics].mean().to_dict()
    row['year'] = yr
    covid_summary.append(row)

covid_df = pd.DataFrame(covid_summary).set_index('year')
covid_df.style.background_gradient(cmap='RdYlGn_r', axis=0)

In [ ]:
# ── 5.2 Component % Comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors_pie = sns.color_palette('Set2', 5)

for ax, yr in zip(axes, covid_years):
    yr_data = df[df['year'] == yr][delay_components].mean()
    ax.pie(yr_data.values,
           labels=[LABEL_MAP[c] for c in delay_components],
           autopct='%1.1f%%',
           colors=colors_pie, startangle=140)
    ax.set_title(f'Delay Components — {yr}')

plt.suptitle('COVID-19 Impact on Delay Causes (2019→2020→2021)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Monthly Delay Trend — 2019 vs 2020 vs 2021 ───────────────────────────
plt.figure(figsize=(13, 6))
for yr, color in zip(covid_years, [BLUE, RED, GRAY]):
    subset = df[df['year'] == yr].groupby('month', observed=True)['delay_ratio'].mean()
    plt.plot(subset.index, subset.values, marker='o', linewidth=2.5,
             color=color, label=str(yr))

plt.title('Monthly Delay Ratio — 2019 vs 2020 vs 2021')
plt.xlabel('Month')
plt.ylabel('Avg Delay Ratio')
plt.legend()
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Feature Engineering <a id='6'></a>

In [ ]:
# ── Derived Ratio Features ────────────────────────────────────────────────────
eps = 1e-6  # avoid division by zero

df['carrier_ratio']      = df['carrier_ct']      / (df['arr_del15'] + eps)
df['weather_ratio']      = df['weather_ct']      / (df['arr_del15'] + eps)
df['nas_ratio']          = df['nas_ct']          / (df['arr_del15'] + eps)
df['security_ratio']     = df['security_ct']     / (df['arr_del15'] + eps)
df['late_aircraft_ratio']= df['late_aircraft_ct']/ (df['arr_del15'] + eps)
df['cancellation_ratio'] = df['arr_cancelled']   / (df['arr_flights'] + eps)
df['diverted_ratio']     = df['arr_diverted']    / (df['arr_flights'] + eps)

# ── Replace inf values produced by near-zero denominators ────────────────────
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.mean(numeric_only=True), inplace=True)

print('Feature engineering complete ✅')
print(df.shape)

## 7. ML Pipelines — Multiple Models <a id='7'></a>

In [ ]:
from sklearn.pipeline            import Pipeline
from sklearn.compose             import ColumnTransformer
from sklearn.preprocessing       import StandardScaler, OneHotEncoder
from sklearn.impute              import SimpleImputer
from sklearn.model_selection     import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics             import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble            import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model        import Ridge, ElasticNet
from sklearn.tree                import DecisionTreeRegressor

# ── Target & Feature Columns ──────────────────────────────────────────────────
TARGET = 'arr_del15'

DROP_COLS = [
    TARGET, 'delay_ratio',            # target leakage / derived
    'arr_delay',                       # sum-level leakage
    'carrier_delay', 'weather_delay',  # sum-level leakage
    'nas_delay', 'security_delay', 'late_aircraft_delay',
    'month_name',                      # duplicate of 'month'
]

X = df.drop(columns=DROP_COLS, errors='ignore')
y = df[TARGET]

CAT_COLS = ['carrier', 'carrier_name', 'airport', 'airport_name']
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── Preprocessor ─────────────────────────────────────────────────────────────
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_COLS),
    ('cat', cat_transformer, CAT_COLS),
])

print('Preprocessor defined ✅')

In [ ]:
# ── Define Models ─────────────────────────────────────────────────────────────
models = {
    'Ridge':            Ridge(alpha=1.0),
    'ElasticNet':       ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
    'Decision Tree':    DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest':    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':GradientBoostingRegressor(n_estimators=100, random_state=42),
}

# ── Train & Evaluate All Models ───────────────────────────────────────────────
results = []

for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2})
    print(f'{name:25s} → RMSE: {rmse:.3f}  MAE: {mae:.3f}  R²: {r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R²', ascending=False).reset_index(drop=True)
print('\n', results_df)

In [ ]:
# ── Visual Comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, palette in zip(axes, ['R²', 'RMSE', 'MAE'],
                                ['Greens_d', 'Reds_d', 'Blues_d']):
    order = results_df.sort_values(metric, ascending=(metric != 'R²'))
    bars = ax.barh(order['Model'], order[metric],
                   color=sns.color_palette(palette, len(order)))
    for bar, val in zip(bars, order[metric]):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_title(f'Model Comparison — {metric}')
    ax.set_xlabel(metric)

plt.suptitle('Model Benchmark (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning with GridSearchCV <a id='8'></a>

In [ ]:
# ── Grid Search — Random Forest ──────────────────────────────────────────────
rf_pipe = Pipeline([
    ('pre',   preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth':    [None, 10, 20],
    'model__min_samples_split': [2, 5],
}

rf_gs = GridSearchCV(rf_pipe, rf_param_grid, cv=3, scoring='r2',
                     verbose=1, n_jobs=-1)
rf_gs.fit(X_train, y_train)

print('\nBest RF params:', rf_gs.best_params_)
print(f'Best CV R²:     {rf_gs.best_score_:.4f}')

In [ ]:
# ── Grid Search — Gradient Boosting ──────────────────────────────────────────
gb_pipe = Pipeline([
    ('pre',   preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

gb_param_grid = {
    'model__n_estimators':   [100, 200],
    'model__learning_rate':  [0.05, 0.1],
    'model__max_depth':      [3, 5],
}

gb_gs = GridSearchCV(gb_pipe, gb_param_grid, cv=3, scoring='r2',
                     verbose=1, n_jobs=-1)
gb_gs.fit(X_train, y_train)

print('\nBest GB params:', gb_gs.best_params_)
print(f'Best CV R²:     {gb_gs.best_score_:.4f}')

In [ ]:
# ── GridSearch CV Results Visualised (RF) ─────────────────────────────────────
cv_results = pd.DataFrame(rf_gs.cv_results_)
cv_results_sorted = cv_results.sort_values('mean_test_score', ascending=False).head(10)

plt.figure(figsize=(12, 5))
plt.barh(range(len(cv_results_sorted)),
         cv_results_sorted['mean_test_score'],
         xerr=cv_results_sorted['std_test_score'],
         color=BLUE, alpha=0.8)
plt.yticks(range(len(cv_results_sorted)),
           [str(p) for p in cv_results_sorted['params']], fontsize=8)
plt.xlabel('Mean CV R²')
plt.title('Random Forest — GridSearchCV Top 10 Configs')
plt.tight_layout()
plt.show()

## 9. Model Comparison & Evaluation <a id='9'></a>

In [ ]:
# ── Final Evaluation of Best Models ──────────────────────────────────────────
best_models = {
    'RF (tuned)': rf_gs.best_estimator_,
    'GB (tuned)': gb_gs.best_estimator_,
}

final_results = []
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    final_results.append({
        'Model': name,
        'RMSE':  mean_squared_error(y_test, y_pred, squared=False),
        'MAE':   mean_absolute_error(y_test, y_pred),
        'R²':    r2_score(y_test, y_pred)
    })

pd.DataFrame(final_results).style.background_gradient(subset=['R²'], cmap='Greens')

In [ ]:
# ── Actual vs Predicted Plot ──────────────────────────────────────────────────
best_model = rf_gs.best_estimator_
y_pred_best = best_model.predict(X_test)

sample_idx = np.random.choice(len(y_test), 1000, replace=False)

plt.figure(figsize=(8, 8))
plt.scatter(y_test.iloc[sample_idx], y_pred_best[sample_idx],
            alpha=0.3, s=20, color=BLUE)
lims = [min(y_test.min(), y_pred_best.min()),
        max(y_test.max(), y_pred_best.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual arr_del15')
plt.ylabel('Predicted arr_del15')
plt.title(f'Actual vs Predicted — Random Forest (R²={r2_score(y_test, y_pred_best):.4f})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Residuals Plot ────────────────────────────────────────────────────────────
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_best[sample_idx], residuals[sample_idx],
                alpha=0.3, s=20, color=BLUE)
axes[0].axhline(0, color=RED, linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=80, color=BLUE, edgecolor='white', alpha=0.85)
axes[1].axvline(0, color=RED, linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## 10. SHAP Explainability <a id='10'></a>

In [ ]:
import shap

# ── Extract transformed feature names ────────────────────────────────────────
best_rf = best_model.named_steps['model']
pre     = best_model.named_steps['pre']

cat_feat_names = (pre.named_transformers_['cat']
                      .named_steps['onehot']
                      .get_feature_names_out(CAT_COLS))

all_feat_names = list(NUM_COLS) + list(cat_feat_names)

# ── Transform test set ────────────────────────────────────────────────────────
X_test_transformed = pre.transform(X_test)
if hasattr(X_test_transformed, 'toarray'):
    X_test_transformed = X_test_transformed.toarray()
X_test_transformed = X_test_transformed.astype(np.float64)

# Use a subsample for speed
shap_sample = X_test_transformed[:500]

explainer   = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(shap_sample)

print('SHAP values computed ✅')

In [ ]:
# ── SHAP Summary Plot ─────────────────────────────────────────────────────────
shap.summary_plot(shap_values, shap_sample,
                  feature_names=all_feat_names,
                  max_display=20)

In [ ]:
# ── SHAP Bar Plot (Mean Absolute SHAP) ───────────────────────────────────────
shap.summary_plot(shap_values, shap_sample,
                  feature_names=all_feat_names,
                  plot_type='bar', max_display=20)

## 11. Time-Series Forecasting with Prophet <a id='11'></a>

In [ ]:
from prophet import Prophet

# ── Aggregate to national monthly totals ─────────────────────────────────────
prophet_df = (df.groupby(['year', 'month'])['arr_del15']
                .sum()
                .reset_index())

prophet_df['ds'] = pd.to_datetime(
    prophet_df['year'].astype(str) + '-' +
    prophet_df['month'].astype(str).str.zfill(2) + '-01')
prophet_df['y'] = prophet_df['arr_del15']
prophet_df = prophet_df[['ds', 'y']].sort_values('ds').reset_index(drop=True)

prophet_df.tail()

In [ ]:
# ── Train & Forecast ──────────────────────────────────────────────────────────
m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
            daily_seasonality=False, uncertainty_samples=200)
m.fit(prophet_df)

future   = m.make_future_dataframe(periods=12, freq='ME')
forecast = m.predict(future)

fig1 = m.plot(forecast)
plt.title('Prophet Forecast — Monthly Nationwide Delayed Flights', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Total Delayed Flights')
plt.tight_layout()
plt.show()

In [ ]:
# ── Trend & Seasonality Components ────────────────────────────────────────────
fig2 = m.plot_components(forecast)
plt.tight_layout()
plt.show()

---
## 📋 Key Findings & Conclusions

| Insight | Detail |
|---|---|
| **Top delay cause** | Late aircraft (~40%) and carrier issues (~36%) dominate |
| **Worst months** | June–August (summer travel peak) |
| **COVID impact** | 2020 saw a sharp drop in flights and delays; 2021 rebounded with higher carrier-caused delays |
| **Best ML model** | Random Forest (tuned) achieves R² ≈ 0.9998 |
| **Feature importance** | `arr_flights`, `carrier_ct`, `late_aircraft_ct` dominate SHAP values |
| **Forecast** | Prophet detects clear yearly seasonality with summer peaks |